# Building Next-Gen RAG with Haystack 2.x

**Advanced NLP Course — Exercise Session**  
Leibniz Universität Hannover

This notebook shows a transition from Naive RAG to an Advanced Hybrid Search pipeline using Haystack’s explicit component graph architecture.

## Part 1: Introduction & Basic Naive RAG
In this section, we instantiate Haystack’s in-memory data store, write our documents, and explicitly connect a Retriever, a PromptBuilder, and a Generator node.

**Step 1: Install Haystack & Core Dependencies**

In [ ]:
!pip install haystack-ai sentence-transformers transformers -q

**Step 2: Define and Index the Knowledge Base**

Haystack relies on an ``InMemoryDocumentStore``. We must first write documents into it before our retriever node can access them.

In [2]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
import json

# 1. Create native Haystack documents

with open("documents.json", "r") as file:
    data = json.load(file)['documents']
documents = [Document(content=json.dumps(d)) for d in data]

# 2. Instantiate memory store
document_store = InMemoryDocumentStore()
document_store.write_documents(documents)

**Step 3: Construct the Naive Graph**

Instead of magic string formatting functions, Haystack uses a generic ``PromptBuilder`` node that consumes variables over the graph connections.

In [14]:
from haystack import Pipeline
from haystack.components.builders import PromptBuilder
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.generators import HuggingFaceLocalGenerator

# 1. Initialize standalone components
retriever = InMemoryBM25Retriever(document_store=document_store, top_k=2)

template = """
Answer the question based only on the following context:
{% for doc in documents %}
    {{ doc.content }}
{% endfor %}

Question: {{ question }}
Answer:"""
prompt_builder = PromptBuilder(template=template)

# Local open-source Generator node
generator = HuggingFaceLocalGenerator(
    model="HuggingFaceTB/SmolLM2-135M-Instruct",
    task="text-generation",
    generation_kwargs={"max_new_tokens": 50, "temperature": 0.2}
)

# 2. Build the structural Pipeline graph
naive_pipeline = Pipeline()
naive_pipeline.add_component("retriever", retriever)
naive_pipeline.add_component("prompt_builder", prompt_builder)
naive_pipeline.add_component("llm", generator)

# 3. Explicitly wire the nodes (Source Node.Output Parameter -> Target Node.Input Parameter)
naive_pipeline.connect("retriever.documents", "prompt_builder.documents")
naive_pipeline.connect("prompt_builder.prompt", "llm.prompt")

output/
/opt/conda/envs/python3/lib/python3.12/site-packages/flashrag/retriever/index_builder.py:124: UserWarning: Some files already exists in save dir and may be overwritten.
  warnings.warn("Some files already exists in save dir and may be overwritten.", UserWarning)
Finish loading...
Loading weights: 100%|████████████████████████| 310/310 [00:08<00:00, 35.46it/s]
/opt/conda/envs/python3/lib/python3.12/site-packages/flashrag/retriever/index_builder.py:538: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  hidden_size = self.encoder.model.get_sentence_embedding_dimension()
/opt/conda/envs/python3/lib/python3.12/site-packages/flashrag/retriever/utils.py:119: UserWarning: Instruction is not set
  warnings.warn('Instruction is not set')
Batches: 100%|████████████████████████████████████| 1/1 [00:12<00:00, 12.30s/it]
Creating index
Finish!


**Step 4: Execute the Naive Pipeline**

In [ ]:
query = "What is the main purpose of the Haystack framework?"

# We feed global entry-point arguments directly into their respective destination inputs
result = naive_pipeline.run({
    "retriever": {"query": query},
    "prompt_builder": {"question": query}
})

print(f"Query: {query}\n")
print(f"Generated Response:\n{result['llm']['replies'][0]}")

## Part 2: Real-World Scenario – Advanced Hybrid Search & Fusion

**Scenario**: Pure keyword search is failing in advanced cases, so we must implement a hybrid approach. In Haystack 2.x, merging distinct sparse (BM25) and dense (Embedding) index lookups is handled via a dedicated `DocumentJoiner` component node right inside the pipeline graph.

**Step 1: Initialize Hybrid Document Store & Embedders**

Because we need vector-based retrieval, we will populate embedding vectors for our text store using a local embedding component.

In [ ]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

# 1. Initialize a vector-capable Document Store
hybrid_store = InMemoryDocumentStore()

# 2. Pre-embed documents to write them into our vector memory partition
doc_embedder = SentenceTransformersDocumentEmbedder(model="intfloat/e5-small-v2")
doc_embedder.warm_up()
docs_with_embeddings = doc_embedder.run(documents)["documents"]
hybrid_store.write_documents(docs_with_embeddings)

**Step 2: Assemble the Advanced Hybrid DAG Pipeline**

In [ ]:
from haystack.components.joiners import DocumentJoiner

# Initialize individual retrieval branches
bm25_branch = InMemoryBM25Retriever(document_store=hybrid_store, top_k=2)
dense_branch = InMemoryEmbeddingRetriever(document_store=hybrid_store, top_k=2)

# Vector query embedder (transforms raw query string to vector format at runtime)
query_embedder = SentenceTransformersTextEmbedder(model="intfloat/e5-small-v2")

# Aggregator node combining outputs using Reciprocal Rank Fusion (RRF)
joiner = DocumentJoiner(join_mode="reciprocal_rank_fusion")

# Re-use our baseline generation definitions from Part 1
adv_prompt_builder = PromptBuilder(template=template)
adv_generator = HuggingFaceLocalGenerator(
    model="HuggingFaceTB/SmolLM2-135M-Instruct",
    task="text-generation",
    generation_kwargs={"max_new_tokens": 60}
)

# --- Construct the advanced graph matrix ---
adv_pipeline = Pipeline()
adv_pipeline.add_component("text_embedder", query_embedder)
adv_pipeline.add_component("bm25_retriever", bm25_branch)
adv_pipeline.add_component("dense_retriever", dense_branch)
adv_pipeline.add_component("doc_joiner", joiner)
adv_pipeline.add_component("prompt_builder", adv_prompt_builder)
adv_pipeline.add_component("llm", adv_generator)

# Link the semantic branch
adv_pipeline.connect("text_embedder.embedding", "dense_retriever.query_embedding")

# Funnel both branches into the joiner node
adv_pipeline.connect("bm25_retriever.documents", "doc_joiner.documents")
adv_pipeline.connect("dense_retriever.documents", "doc_joiner.documents")

# Feed merged documents forward to generation
adv_pipeline.connect("doc_joiner.documents", "prompt_builder.documents")
adv_pipeline.connect("prompt_builder.prompt", "llm.prompt")


**Step 3: Run the Advanced Real-World Scenario**

In [ ]:
complex_query = "Explain advanced RAG systems in the current year 2026."

adv_output = adv_pipeline.run({
    "text_embedder": {"text": complex_query},
    "bm25_retriever": {"query": complex_query},
    "prompt_builder": {"question": complex_query}
})

print(f"Resulting Answer:\n{adv_output['llm']['replies'][0]}")

---
*Advanced NLP · Exercise Session · SoSe 2026*